# 🩺 Daily Challenge: Breast Cancer Prediction

In this notebook we explore the **Breast Cancer Wisconsin (Diagnostic)** dataset and build four classification models — **Logistic Regression**, **K-Nearest Neighbours**, **Random Forest**, and **Support Vector Machine (SVM)** — to predict whether a tumor is **Malignant (M)** or **Benign (B)**.

**Dataset:** [Kaggle - Breast Cancer Wisconsin Data](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)

> 📁 **Before running:** Download `data.csv` from the Kaggle link above and upload it to your Colab session (or mount Google Drive), so the path `data.csv` resolves correctly. In Colab you can run:
> ```python
> from google.colab import files
> uploaded = files.upload()  # then select data.csv
> ```


In [4]:
# 📦 Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_style("whitegrid")
%matplotlib inline


ModuleNotFoundError: No module named 'sklearn'

## 1. Exploratory Data Analysis (EDA)

### 1.1 Load the dataset and examine the first few rows

In [ ]:
df = pd.read_csv("data.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


### 1.2 Check and handle missing values

In [ ]:
# Check for missing values in each column
df.isnull().sum()


In [ ]:
# The Kaggle version of this dataset typically has a fully-empty 'Unnamed: 32' column
# (an artifact of a trailing comma in the CSV). Let's confirm and check overall missingness.
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
missing_pct[missing_pct > 0]


### 1.3 Drop unnecessary columns

`id` is just a row identifier with no predictive value, and `Unnamed: 32` is an empty artifact column — both are dropped.

In [ ]:
cols_to_drop = [c for c in ["id", "Unnamed: 32"] if c in df.columns]
df = df.drop(columns=cols_to_drop)

# Double check there are no remaining missing values
print("Remaining missing values:\n", df.isnull().sum().sum())
df.head()


### 1.4 Countplot of `diagnosis` (magma palette)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="diagnosis", data=df, palette="magma")
plt.title("Count of Diagnosis (M = Malignant, B = Benign)")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.show()


## 2. Data Preprocessing, Building Models and Evaluation

### 2.1 Counts of unique values in `diagnosis`

In [ ]:
df["diagnosis"].value_counts()


### 2.2 Map categorical values to numerical values

We encode `M` (Malignant) as `1` and `B` (Benign) as `0`.

In [ ]:
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
df["diagnosis"].value_counts()


### 2.3 Split the data into train and test sets

In [ ]:
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features - important for KNN, SVM and (helpful for) Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


### 2.4 Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=10000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Accuracy: {acc_lr:.4f}")
print(classification_report(y_test, y_pred_lr))


### 2.5 K Nearest Neighbours

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

acc_knn = accuracy_score(y_test, y_pred_knn)
print(f"K Nearest Neighbours Accuracy: {acc_knn:.4f}")
print(classification_report(y_test, y_pred_knn))


### 2.6 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)  # Random Forest doesn't strictly need scaling
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {acc_rf:.4f}")
print(classification_report(y_test, y_pred_rf))


### 2.7 Support Vector Machine (SVM)

In [ ]:
svm = SVC(kernel="rbf", random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)

acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"SVM Accuracy: {acc_svm:.4f}")
print(classification_report(y_test, y_pred_svm))


### 2.8 Model Comparison — Which is the best model?

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "K Nearest Neighbours", "Random Forest", "SVM"],
    "Accuracy": [acc_lr, acc_knn, acc_rf, acc_svm]
}).sort_values("Accuracy", ascending=False).reset_index(drop=True)

results


In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x="Accuracy", y="Model", data=results, palette="magma")
plt.xlim(0.8, 1.0)
plt.title("Model Accuracy Comparison")
plt.show()

best_model = results.iloc[0]
print(f"🏆 Best model: {best_model['Model']} with accuracy of {best_model['Accuracy']:.4f}")


### 📌 Conclusion

All four models tend to perform strongly on this dataset (typically 95–99% accuracy) because the features are well-separated between malignant and benign tumors. In practice:

- **Logistic Regression** and **SVM** (with scaled features) often perform best on this dataset since the classes are largely linearly separable.
- **Random Forest** is usually close behind and is robust to unscaled features and outliers.
- **KNN** can be sensitive to the choice of `k` and feature scaling, but still performs competitively here.

The exact "best" model depends on the random train/test split and hyperparameters — re-run with different `random_state` values or try cross-validation / `GridSearchCV` for a more robust comparison.

**⚠️ A note on this domain:** This is a great exercise for learning classification techniques, but real diagnostic decisions should always be made by qualified medical professionals using validated clinical tools — these models are for educational purposes only.
